# Objective 2 - Linear Regression

Sairaj Jaspalsinh Rajput - S397925

Kiran Subedi - S400572

# Part 1 – Data preparation (Sairaj)

In [7]:
import random
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

PERMA_SEED = 42
random.seed(PERMA_SEED)
np.random.seed(PERMA_SEED)

conn = sqlite3.connect("../db/fifa2026.db")

matches = pd.read_sql("SELECT * FROM matches", conn)
teams = pd.read_sql("SELECT * FROM teams", conn)
squads = pd.read_sql("SELECT * FROM squads_and_players", conn)
venues = pd.read_sql("SELECT * FROM venues", conn)
stages = pd.read_sql("SELECT * FROM tournament_stages", conn)

print(matches.shape, teams.shape, squads.shape, venues.shape, stages.shape)


(104, 17) (48, 8) (1248, 10) (16, 8) (7, 3)


In [8]:
# age of each player on the opening day
squads["age"] = (pd.to_datetime("2026-06-11") - pd.to_datetime(squads["date_of_birth"])).dt.days / 365.25

# one row per team
team_info = squads.groupby("team_id").agg(
    squad_value=("market_value_eur", "sum"),
    avg_age=("age", "mean"),
    total_caps=("caps", "sum"),
    avg_height=("height_cm", "mean"),
).reset_index()

# goals scored by each team's forwards
fwd_goals = squads[squads["position"] == "FWD"].groupby("team_id")["goals"].sum().reset_index(name="fwd_goals")

team_info = team_info.merge(fwd_goals, on="team_id")
team_info = team_info.merge(teams[["team_id", "team_name", "fifa_code", "elo_rating"]], on="team_id")

print(team_info.shape)
team_info.sort_values("squad_value", ascending=False).head()

(48, 9)


,team_id,squad_value,avg_age,total_caps,avg_height,fwd_goals,team_name,fifa_code,elo_rating
28,29,2134173029,26.747433,758,181.730769,78,Spain,ESP,2120
32,33,1546204469,27.001000,816,184.923077,83,France,FRA,2100
44,45,1321818896,27.273206,853,184.615385,126,England,ENG,2050
40,41,1145908765,28.035276,1176,181.500000,188,Portugal,POR,2010
36,37,1025039444,29.118201,1267,179.576923,179,Argentina,ARG,2150


In [9]:
# add stages (group/knockout) and venue details to each match
matches = matches.merge(stages[["stage_id", "is_knockout"]], on="stage_id")
matches = matches.merge(venues[["venue_id", "country", "elevation_meters"]], on="venue_id")
matches["kickoff"] = pd.to_datetime(matches["date"] + " " + matches["kickoff_time_utc"])

# each match has two teams, so making one row for each team
home = pd.DataFrame({
    "match_id": matches["match_id"], "kickoff": matches["kickoff"],
    "team_id": matches["home_team_id"], "opp_id": matches["away_team_id"],
    "goals_for": matches["home_score"], "goals_against": matches["away_score"],
    "xg_for": matches["home_xg"], "xg_against": matches["away_xg"],
    "is_home": 1, "is_knockout": matches["is_knockout"],
    "venue_country": matches["country"], "elevation": matches["elevation_meters"],
})
away = pd.DataFrame({
    "match_id": matches["match_id"], "kickoff": matches["kickoff"],
    "team_id": matches["away_team_id"], "opp_id": matches["home_team_id"],
    "goals_for": matches["away_score"], "goals_against": matches["home_score"],
    "xg_for": matches["away_xg"], "xg_against": matches["home_xg"],
    "is_home": 0, "is_knockout": matches["is_knockout"],
    "venue_country": matches["country"], "elevation": matches["elevation_meters"],
})

team_matches = pd.concat([home,away]).sort_values(["team_id", "kickoff"]).reset_index(drop=True)
print(team_matches.shape)


(208, 12)


## Part 2 – LR 2.1: Predicting goal difference (Kiran)
One row per match (104 rows). Target = home goals − away goals.
Each feature is **home team value − away team value**, except `is_knockout`.

| Feature | Meaning |
|---|---|
| elo_diff | Difference in pre-tournament Elo rating |
| value_diff | log(home squad value) − log(away squad value) |
| age_diff | Difference in average squad age |
| fwd_goals_diff | Difference in forwards' international goals |
| prev_xg_diff | Difference in xG form (xG for − against) from earlier matches |
| rest_diff | Difference in rest days |
| host_diff | Host nation playing at home |
| is_knockout | 1 if knockout stage |

In [ ]:
# Separate home and away team rows so they can be compared by match.
h = team_matches[team_matches["is_home"] == 1].set_index("match_id").sort_index()
a = team_matches[team_matches["is_home"] == 0].set_index("match_id").sort_index()

# Calculate home-minus-away differences for each feature and the match result.
lr1 = pd.DataFrame({
    "elo_diff": h["elo_rating"] - a["elo_rating"],
    "value_diff": np.log(h["squad_value"]) - np.log(a["squad_value"]),
    "age_diff": h["avg_age"] - a["avg_age"],
    "fwd_goals_diff": h["fwd_goals"] - a["fwd_goals"],
    "prev_xg_diff": (h["prev_xg_for"] - h["prev_xg_against"]) - (a["prev_xg_for"] - a["prev_xg_against"]),
    "rest_diff": h["rest_days"] - a["rest_days"],
    "host_diff": h["is_host"] - a["is_host"],
    "is_knockout": h["is_knockout"],
    "goal_difference": h["goals_for"] - h["goals_against"],
}).reset_index()

# List the input features that will be used to predict goal difference.
features1 = ["elo_diff", "value_diff", "age_diff", "fwd_goals_diff",
             "prev_xg_diff", "rest_diff", "host_diff", "is_knockout"]

# Check the dataset size and count any missing values.
print(lr1.shape)
print("Missing values:", lr1.isna().sum().sum())
lr1.head()